# Image Search — Faceted (CLIP + Color)

A single interactive panel that combines:

- **CLIP text query** — semantic search ("a portrait", "misty forest", ...)
- **Dominant color picker** — pick a colour, find images whose dominant colour is close to it
- **Hue / Saturation / Brightness range sliders** — hard filters on pixel statistics
- **Semantic ↔ Color weight** — slide between "rank by meaning" and "rank by colour match"

Built on the same primitives as the toolkit notebook (CLIP embeddings + pixel features), but with one dominant-RGB colour added per image (computed once, cached).

> Tip: leave any control at its default to disable it. Only enabled facets affect the result.

## 1. Install dependencies

Run once. On Apple Silicon, PyTorch will use the MPS backend automatically.

In [1]:
%pip install torch torchvision open_clip_torch pillow numpy scikit-learn matplotlib tqdm ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & config

In [2]:
import os, math, hashlib, pickle, colorsys, zipfile
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import open_clip
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

# === EDIT THIS ===
IMAGE_FOLDER = Path('./Pictures')
CACHE_DIR    = Path('./cache')
DOWNLOAD_DIR = Path('./downloads')
CACHE_DIR.mkdir(exist_ok=True)
DOWNLOAD_DIR.mkdir(exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available()
          else 'cpu')
print('device:', device)

# placeholders so later cells can be safely re-run
model = tokenizer = preprocess = None
encode_text = encode_images = None
MODEL_NAME = PRETRAINED = None

device: cpu


## 3. Pre-flight — clean the image folder

Moves anything that shouldn't be processed into a sibling `*_dump/` folder:

- files whose extension isn't in `IMG_EXTS`
- files PIL can't open (corrupt / truncated)
- exact duplicates (same bytes, MD5-matched) — keeps the first occurrence

Run once before the scan. Re-running is safe: the dump folder lives outside `IMAGE_FOLDER` so it's never re-scanned.

In [3]:
import shutil

DUMP_DIR = IMAGE_FOLDER.parent / f'{IMAGE_FOLDER.name}_dump'
DUMP_DIR.mkdir(exist_ok=True)

def _safe_move(src, dest_dir):
    dest = dest_dir / src.name
    n = 1
    while dest.exists():
        dest = dest_dir / f'{src.stem}__{n}{src.suffix}'
        n += 1
    shutil.move(str(src), str(dest))
    return dest

def _file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for buf in iter(lambda: f.read(chunk), b''):
            h.update(buf)
    return h.hexdigest()

bad_ext, bad_open, dupes = [], [], []
seen = {}  # md5 -> kept path

all_files = sorted(p for p in IMAGE_FOLDER.rglob('*') if p.is_file())
print(f'scanning {len(all_files)} files in {IMAGE_FOLDER}/ ...')

for p in tqdm(all_files):
    # 1. wrong extension
    if p.suffix.lower() not in IMG_EXTS:
        bad_ext.append(_safe_move(p, DUMP_DIR))
        continue
    # 2. unreadable / corrupt
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        bad_open.append(_safe_move(p, DUMP_DIR))
        continue
    # 3. exact duplicate (same bytes)
    h = _file_md5(p)
    if h in seen:
        dupes.append((_safe_move(p, DUMP_DIR), seen[h].name))
    else:
        seen[h] = p

print(f'\nmoved to {DUMP_DIR}/:')
print(f'  • {len(bad_ext):>4} non-image extensions')
print(f'  • {len(bad_open):>4} unreadable / corrupt files')
print(f'  • {len(dupes):>4} exact duplicates')
print(f'kept in {IMAGE_FOLDER}/: {len(seen)} unique images')
if dupes:
    print('\nfirst few duplicates (moved → kept):')
    for moved, kept in dupes[:5]:
        print(f'  {moved.name}  →  {kept}')

scanning 442 files in Pictures/ ...


  0%|          | 0/442 [00:00<?, ?it/s]


moved to Pictures_dump/:
  •    0 non-image extensions
  •    0 unreadable / corrupt files
  •    0 exact duplicates
kept in Pictures/: 442 unique images


## 4. Pick a CLIP model

| Preset           | Size  | Quality | Speed  | Notes                              |
|------------------|-------|---------|--------|------------------------------------|
| ViT-B-32 (laion) | ~150M | ok      | fast   | Default — good for quick iteration |
| ViT-B-16 (laion) | ~150M | better  | medium | Better fine detail than B-32       |
| ViT-L-14 (laion) | ~430M | strong  | slow   | The "good" one for serious search  |
| ViT-L-14 (datacomp) | ~430M | strong | slow | Often beats laion L-14 on retrieval|
| ViT-H-14 (laion) | ~1B   | best    | heavy  | Needs a beefy GPU                  |
| ViT-bigG-14 (laion) | ~2.5B | best+ | very heavy | Only if you have lots of VRAM   |

The cache key embeds the model name, so switching models reuses old caches and only recomputes what's missing. **Click the button** to load the model — switching the dropdown alone does nothing until you click.

In [4]:
import ipywidgets as widgets
from IPython.display import display

# (model_name, pretrained_tag) — see open_clip.list_pretrained() for more
MODEL_PRESETS = {
    'ViT-B-32 · laion2b (fast, default)':   ('ViT-B-32',     'laion2b_s34b_b79k'),
    'ViT-B-16 · laion2b':                   ('ViT-B-16',     'laion2b_s34b_b88k'),
    'ViT-L-14 · laion2b (recommended)':     ('ViT-L-14',     'laion2b_s32b_b82k'),
    'ViT-L-14 · datacomp_xl':               ('ViT-L-14',     'datacomp_xl_s13b_b90k'),
    'ViT-H-14 · laion2b (heavy)':           ('ViT-H-14',     'laion2b_s32b_b79k'),
    'ViT-bigG-14 · laion2b (very heavy)':   ('ViT-bigG-14',  'laion2b_s39b_b160k'),
}

model_dd  = widgets.Dropdown(options=list(MODEL_PRESETS), value=list(MODEL_PRESETS)[0],
                             description='Model:', layout=widgets.Layout(width='60%'))
load_btn  = widgets.Button(description='Load model', button_style='primary')
load_out  = widgets.Output()

def load_model(_=None):
    global model, tokenizer, preprocess, encode_text, encode_images
    global MODEL_NAME, PRETRAINED
    load_out.clear_output()
    with load_out:
        MODEL_NAME, PRETRAINED = MODEL_PRESETS[model_dd.value]
        print(f'loading {MODEL_NAME} / {PRETRAINED} on {device} ...')
        hf_cache = os.path.abspath('./hf_cache'); os.makedirs(hf_cache, exist_ok=True)
        model, _, preprocess = open_clip.create_model_and_transforms(
            MODEL_NAME, pretrained=PRETRAINED, cache_dir=hf_cache,
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer(MODEL_NAME)

        @torch.no_grad()
        def _encode_text(texts):
            if isinstance(texts, str):
                texts = [texts]
            tok = tokenizer(texts).to(device)
            f = model.encode_text(tok).float()
            f /= f.norm(dim=-1, keepdim=True)
            return f.cpu().numpy()

        @torch.no_grad()
        def _encode_images(pil_images):
            batch = torch.stack([preprocess(im) for im in pil_images]).to(device)
            f = model.encode_image(batch).float()
            f /= f.norm(dim=-1, keepdim=True)
            return f.cpu().numpy()

        encode_text, encode_images = _encode_text, _encode_images
        print(f'OK — model loaded. Now run the next cell to build/load the index.')

load_btn.on_click(load_model)
display(widgets.VBox([widgets.HBox([model_dd, load_btn]), load_out]))

## 5. Build / load index

For each image we cache:
- CLIP embedding (depends on which model you loaded above — switching the model triggers a full recompute)
- Pixel features: `(brightness, hue, saturation, complexity)` — same as the toolkit
- **Dominant RGB color** (3 floats, 0–1) — k-means on a 64×64 thumbnail, centroid of the largest cluster

Cache key includes folder + file list + file sizes + model name, so switching models recomputes embeddings but switching back reuses the old cache. Edit a file and it recomputes automatically.

In [5]:
def list_images(folder):
    return sorted([p for p in Path(folder).rglob('*') if p.suffix.lower() in IMG_EXTS])

def cache_key(paths):
    h = hashlib.md5()
    for p in paths:
        h.update(str(p).encode())
        try:
            h.update(str(p.stat().st_size).encode())
        except FileNotFoundError:
            pass
    h.update(MODEL_NAME.encode())
    h.update(b'v2-with-dominant-rgb')   # bump if format changes
    return h.hexdigest()[:16]

def compute_pixel_features(pil):
    """brightness, hue, saturation, complexity — all 0..1."""
    small = pil.convert('RGB').resize((64, 64))
    arr = np.asarray(small).astype(np.float32) / 255.0
    brightness = float(0.2126*arr[..., 0].mean() + 0.7152*arr[..., 1].mean() + 0.0722*arr[..., 2].mean())
    hsv = np.asarray(small.convert('HSV')).astype(np.float32) / 255.0
    weights = hsv[..., 1].flatten() + 1e-6
    angles = hsv[..., 0].flatten() * 2*np.pi
    x = np.average(np.cos(angles), weights=weights)
    y = np.average(np.sin(angles), weights=weights)
    hue = (math.atan2(y, x) / (2*np.pi)) % 1.0
    sat = float(hsv[..., 1].mean())
    gray = arr.mean(-1)
    gx = np.abs(np.diff(gray, axis=1))
    gy = np.abs(np.diff(gray, axis=0))
    complexity = float((gx.mean() + gy.mean()) / 2)
    return brightness, hue, sat, complexity

def compute_dominant_rgb(pil, k=5):
    """Largest k-means cluster centroid on a 64x64 thumbnail. Returns (r,g,b) in 0..1."""
    small = pil.convert('RGB').resize((64, 64))
    pixels = np.asarray(small).reshape(-1, 3).astype(np.float32) / 255.0
    km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(pixels)
    sizes = np.bincount(km.labels_, minlength=k)
    return tuple(km.cluster_centers_[int(np.argmax(sizes))])

def build_index(folder, batch_size=32):
    if encode_images is None or MODEL_NAME is None:
        raise RuntimeError('Load a CLIP model first (run section 4 and click "Load model").')
    paths = list_images(folder)
    if not paths:
        raise RuntimeError(f'No images found in {folder}')
    key = cache_key(paths)
    cache_file = CACHE_DIR / f'faceted_index_{key}.pkl'
    if cache_file.exists():
        print(f'loading cached index: {cache_file.name}')
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    print(f'computing embeddings + features for {len(paths)} images...')
    embs, pix, doms, valid = [], [], [], []
    for i in tqdm(range(0, len(paths), batch_size)):
        chunk = paths[i:i+batch_size]
        imgs = []
        for p in chunk:
            try:
                im = Image.open(p).convert('RGB')
                imgs.append(im)
                pix.append(compute_pixel_features(im))
                doms.append(compute_dominant_rgb(im))
                valid.append(p)
            except Exception as e:
                print('skip', p, e)
        if imgs:
            embs.append(encode_images(imgs))
    index = {
        'paths': valid,
        'embeddings': np.vstack(embs),
        'pixel_feats': np.array(pix),
        'dominant_rgb': np.array(doms),
    }
    with open(cache_file, 'wb') as f:
        pickle.dump(index, f)
    print(f'cached to {cache_file}')
    return index

index         = build_index(IMAGE_FOLDER)
paths         = index['paths']
embeddings    = index['embeddings']
pixel_feats   = index['pixel_feats']        # cols: brightness, hue, sat, complexity
dominant_rgb  = index['dominant_rgb']       # (N, 3) in 0..1
print('images:', len(paths), '| embedding dim:', embeddings.shape[1])

loading cached index: faceted_index_03eb659c8b7390e4.pkl
images: 434 | embedding dim: 768


## 6. Display + download helpers

Each thumbnail shows the **filename** and an optional score line underneath. A colour swatch bar shows the dominant colour.

`zip_results()` packs matched files into `./downloads/` with optional score prefixes in the archive filenames. `make_download_button()` wires a button to a state dict so each panel gets its own **Download as ZIP** button.

In [6]:
from IPython.display import FileLink

def _short(name, n=22):
    return name if len(name) <= n else name[:n-1] + '…'

def _safe_prefix(s):
    return ''.join(c if c.isalnum() or c in '.+-' else '_' for c in str(s))

def show_grid_with_swatch(image_paths, swatches, titles=None, cols=5, thumb=180,
                          figtitle=None, progress_cb=None, show_filenames=True):
    n = len(image_paths)
    if n == 0:
        print('(no results)'); return
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*thumb/55, rows*thumb/45))
    axes = np.array(axes).reshape(rows, cols)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i < n:
            try:
                im = Image.open(image_paths[i]).convert('RGB')
                im.thumbnail((thumb, thumb))
                ax.imshow(im)
                rgb = swatches[i]
                ax.add_patch(Rectangle((0, -0.08*thumb), thumb, 0.06*thumb,
                                        transform=ax.transData,
                                        facecolor=rgb, edgecolor='black',
                                        linewidth=0.5, clip_on=False))
                parts = []
                if titles is not None and i < len(titles) and titles[i]:
                    parts.append(str(titles[i])[:40])
                if show_filenames:
                    parts.append(_short(Path(image_paths[i]).name))
                if parts:
                    ax.set_title('\n'.join(parts), fontsize=7)
                if progress_cb:
                    progress_cb(i + 1, n)
            except Exception:
                ax.set_title('err', fontsize=7)
    if figtitle:
        fig.suptitle(figtitle, fontsize=12)
    plt.tight_layout(); plt.show()

def zip_results(paths_list, tag='results', prefixes=None):
    if not paths_list:
        print('nothing to download'); return None
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    safe_tag = ''.join(c if c.isalnum() or c in '-_' else '_' for c in tag)[:40]
    zp = DOWNLOAD_DIR / f'{safe_tag}_{stamp}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:
        seen = {}
        for i, p in enumerate(paths_list):
            base = Path(p).name
            if prefixes is not None and i < len(prefixes):
                base = f'{_safe_prefix(prefixes[i])}{base}'
            if base in seen:
                seen[base] += 1
                stem, suf = Path(base).stem, Path(base).suffix
                arc = f'{stem}_{seen[base]}{suf}'
            else:
                seen[base] = 0
                arc = base
            z.write(p, arcname=arc)
    return zp

def make_download_button(state, label='Download as ZIP'):
    btn = widgets.Button(description=label, icon='download')
    dl_out = widgets.Output()
    def _on(_):
        dl_out.clear_output()
        with dl_out:
            p_list = state.get('paths', [])
            if not p_list: print('Run a search first.'); return
            tag = state.get('tag', 'results')
            prefixes = state.get('prefixes')
            zp = zip_results(p_list, tag=tag, prefixes=prefixes)
            if zp is not None:
                print(f'{len(p_list)} files → {zp}  ({zp.stat().st_size/1e6:.1f} MB)')
                display(FileLink(str(zp)))
    btn.on_click(_on)
    return widgets.VBox([btn, dl_out])

## 7. Faceted search — the interactive panel

**Three things to know:**

- **Per-facet toggles.** Each facet has its own on/off checkbox. Unchecked facets are skipped entirely — controls are greyed out to make it visually clear.
- **Search on click only.** Nothing runs until you press **Search**. Changing sliders or typing does not trigger a search.
- **Progress bar.** Shows staged progress (encoding → scoring → filtering → ranking → thumbnails). Thumbnail loading is the slowest stage in practice, so most of the bar's travel happens there.

**Scoring rules:**

| Text | Color | Result                                                                  |
|------|-------|-------------------------------------------------------------------------|
| ☑    | ☐     | Pure CLIP ranking                                                       |
| ☐    | ☑     | Pure color ranking (closeness of dominant RGB to the picked colour)     |
| ☑    | ☑     | Weighted combination — the **Text ↔ Color** slider controls the blend   |
| ☐    | ☐     | No ranking — just shows the first K images that pass the hard filters   |

**Hard filters** (brightness, saturation, hue) are always AND-combined with the ranking — disable them to skip. Hue range supports wraparound (e.g. `0.95 → 0.05` for reds).

In [7]:
import ipywidgets as widgets
from IPython.display import display

# ---- scoring helpers ----
def clip_scores(query):
    q = encode_text(query)[0]
    s = embeddings @ q
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-9)

def color_scores(target_rgb):
    t = np.array(target_rgb, dtype=np.float32)
    d = np.linalg.norm(dominant_rgb - t, axis=1) / math.sqrt(3.0)
    return 1.0 - d

def hex_to_rgb(hex_color):
    h = hex_color.lstrip('#')
    return tuple(int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))

def hue_mask(lo, hi, hues):
    return (hues >= lo) & (hues <= hi) if lo <= hi else (hues >= lo) | (hues <= hi)

# ---- state dict for download ----
facet_state = {'paths': [], 'prefixes': [], 'tag': 'faceted'}

# ---- controls ----
use_text   = widgets.Checkbox(value=True,  description='Text',       indent=False, layout=widgets.Layout(width='110px'))
query_box  = widgets.Text(value='', placeholder='e.g. a portrait of a person',
                          layout=widgets.Layout(width='70%'))

use_color  = widgets.Checkbox(value=False, description='Color',      indent=False, layout=widgets.Layout(width='110px'))
color_pick = widgets.ColorPicker(concise=False, value='#ff6633', disabled=True,
                                 layout=widgets.Layout(width='30%'))

color_weight = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                   description='Text ↔ Color', readout_format='.2f',
                                   layout=widgets.Layout(width='55%'))

use_bright   = widgets.Checkbox(value=False, description='Brightness', indent=False, layout=widgets.Layout(width='110px'))
bright_range = widgets.FloatRangeSlider(value=[0.0, 1.0], min=0.0, max=1.0, step=0.05,
                                        readout_format='.2f', disabled=True,
                                        layout=widgets.Layout(width='55%'))

use_sat   = widgets.Checkbox(value=False, description='Saturation', indent=False, layout=widgets.Layout(width='110px'))
sat_range = widgets.FloatRangeSlider(value=[0.0, 1.0], min=0.0, max=1.0, step=0.05,
                                     readout_format='.2f', disabled=True,
                                     layout=widgets.Layout(width='55%'))

use_hue   = widgets.Checkbox(value=False, description='Hue',        indent=False, layout=widgets.Layout(width='110px'))
hue_range = widgets.FloatRangeSlider(value=[0.0, 1.0], min=0.0, max=1.0, step=0.02,
                                     readout_format='.2f', disabled=True,
                                     layout=widgets.Layout(width='55%'))

topk    = widgets.IntSlider(value=12, min=1, max=40, description='Top K')
run_btn = widgets.Button(description='Search', button_style='primary',
                         icon='search', layout=widgets.Layout(width='150px'))

progress = widgets.IntProgress(value=0, min=0, max=100, bar_style='',
                               layout=widgets.Layout(width='50%'))
status   = widgets.Label(value='Ready — configure facets and press Search')
out = widgets.Output()

def _bind_enable(cb, *targets):
    def _h(change):
        for t in targets: t.disabled = not change['new']
    cb.observe(_h, names='value')

_bind_enable(use_text,   query_box)
_bind_enable(use_color,  color_pick)
_bind_enable(use_bright, bright_range)
_bind_enable(use_sat,    sat_range)
_bind_enable(use_hue,    hue_range)

def do_search(_=None):
    out.clear_output()
    progress.bar_style = 'info'; progress.value = 0
    status.value = 'Starting...'; run_btn.disabled = True
    try:
        with out:
            if 'embeddings' not in globals() or 'dominant_rgb' not in globals():
                print('Run the model + index cells first (sections 4 & 5).'); return

            have_text  = use_text.value  and query_box.value.strip() != ''
            have_color = use_color.value
            scores = None; mode_parts = []

            if have_text:
                progress.value = 10; status.value = 'Encoding text query...'
                ts = clip_scores(query_box.value.strip())
                mode_parts.append(f'"{query_box.value.strip()}"')
            else:
                ts = None
            if have_color:
                progress.value = 25; status.value = 'Computing color scores...'
                cs = color_scores(hex_to_rgb(color_pick.value))
                mode_parts.append(f'color {color_pick.value}')
            else:
                cs = None

            if ts is not None and cs is not None:
                cw = color_weight.value
                scores = (1 - cw) * ts + cw * cs
                mode_parts[-1] += f' (w={cw:.2f})'
            elif ts is not None: scores = ts
            elif cs is not None: scores = cs

            progress.value = 40; status.value = 'Applying hard filters...'
            mask = np.ones(len(paths), dtype=bool)
            if use_bright.value:
                b = pixel_feats[:, 0]
                mask &= (b >= bright_range.value[0]) & (b <= bright_range.value[1])
                mode_parts.append(f'bright {bright_range.value[0]:.2f}-{bright_range.value[1]:.2f}')
            if use_sat.value:
                sa = pixel_feats[:, 2]
                mask &= (sa >= sat_range.value[0]) & (sa <= sat_range.value[1])
                mode_parts.append(f'sat {sat_range.value[0]:.2f}-{sat_range.value[1]:.2f}')
            if use_hue.value:
                mask &= hue_mask(hue_range.value[0], hue_range.value[1], pixel_feats[:, 1])
                mode_parts.append(f'hue {hue_range.value[0]:.2f}-{hue_range.value[1]:.2f}')

            n_kept = int(mask.sum())
            if n_kept == 0:
                status.value = 'No images pass the hard filters'
                progress.bar_style = 'warning'
                print('Loosen one of the range filters.'); return

            progress.value = 50; status.value = 'Ranking...'
            if scores is None:
                idx = np.where(mask)[0][:topk.value].tolist()
                titles_out = [''] * len(idx)
                if not mode_parts: mode_parts.append('(no facets — showing first K)')
            else:
                masked = np.where(mask, scores, -np.inf)
                idx = np.argsort(-masked)[:topk.value]
                idx = [int(i) for i in idx if np.isfinite(masked[i])]
                titles_out = [f'{scores[i]:.3f}' for i in idx]

            if not idx:
                status.value = 'No matches'; progress.bar_style = 'warning'
                print('No matches.'); return

            # update state for download
            facet_state['paths']    = [paths[i] for i in idx]
            facet_state['prefixes'] = [f'{scores[i]:.3f}_' for i in idx] if scores is not None else None
            facet_state['tag']      = '_'.join(mode_parts)[:60]

            print(f'{n_kept} of {len(paths)} images pass filters; showing top {len(idx)}')

            def img_progress(done, total):
                progress.value = 50 + int(50 * done / total)
                status.value = f'Loading thumbnail {done}/{total}'

            swatches = [tuple(dominant_rgb[i]) for i in idx]
            show_grid_with_swatch(
                [paths[i] for i in idx], swatches=swatches,
                titles=titles_out, cols=min(5, len(idx)), thumb=180,
                figtitle=' | '.join(mode_parts), progress_cb=img_progress,
            )
            progress.value = 100; progress.bar_style = 'success'
            status.value = f'Done — {len(idx)} matches'
    except Exception as e:
        progress.bar_style = 'danger'; status.value = f'Error: {e}'; raise
    finally:
        run_btn.disabled = False

run_btn.on_click(do_search)

def _row(cb, control):
    return widgets.HBox([cb, control])

dl_facet = make_download_button(facet_state)

display(widgets.VBox([
    _row(use_text,   query_box),
    _row(use_color,  color_pick),
    color_weight,
    _row(use_bright, bright_range),
    _row(use_sat,    sat_range),
    _row(use_hue,    hue_range),
    widgets.HBox([topk, run_btn]),
    widgets.HBox([progress, status]),
    out,
    dl_facet,
]))

## 8. Bonus — Color story (interactive)

Run a facet search just like the main panel, then re-sort the top N results **left-to-right by hue** so you see the colour progression of the matches. Useful when you've found a thematic set and want to lay it out as a colour sequence.

In [8]:
import ipywidgets as widgets
from IPython.display import display

def _bind_disabled(cb, *targets):
    def _h(change):
        for t in targets: t.disabled = not change['new']
    cb.observe(_h, names='value')

cs_state = {'paths': [], 'prefixes': [], 'tag': 'color_story'}

cs_use_text  = widgets.Checkbox(value=True,  description='Text',  indent=False, layout=widgets.Layout(width='110px'))
cs_query     = widgets.Text(value='a portrait', placeholder='semantic query',
                            layout=widgets.Layout(width='70%'))
cs_use_color = widgets.Checkbox(value=True,  description='Color', indent=False, layout=widgets.Layout(width='110px'))
cs_color     = widgets.ColorPicker(concise=False, value='#cc7744',
                                   layout=widgets.Layout(width='30%'))
cs_weight    = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05,
                                   description='Text ↔ Color', readout_format='.2f',
                                   layout=widgets.Layout(width='55%'))
cs_n         = widgets.IntSlider(value=15, min=3, max=40, description='N')
cs_btn       = widgets.Button(description='Build color story', button_style='primary',
                              icon='play', layout=widgets.Layout(width='200px'))
cs_prog      = widgets.IntProgress(value=0, min=0, max=100, bar_style='',
                                   layout=widgets.Layout(width='50%'))
cs_status    = widgets.Label(value='Ready')
cs_out       = widgets.Output()

_bind_disabled(cs_use_text,  cs_query)
_bind_disabled(cs_use_color, cs_color)

def run_color_story(_=None):
    cs_out.clear_output()
    cs_prog.bar_style = 'info'; cs_prog.value = 0
    cs_status.value = 'Starting...'; cs_btn.disabled = True
    try:
        with cs_out:
            if 'embeddings' not in globals():
                print('Build the index first (sections 4 & 5).'); return
            have_text  = cs_use_text.value  and cs_query.value.strip() != ''
            have_color = cs_use_color.value
            if not have_text and not have_color:
                print('Enable at least one of Text or Color.'); return

            ts = cs_scores_arr = None
            if have_text:
                cs_prog.value = 15; cs_status.value = 'Encoding text...'
                ts = clip_scores(cs_query.value.strip())
            if have_color:
                cs_prog.value = 30; cs_status.value = 'Computing color scores...'
                cs_scores_arr = color_scores(hex_to_rgb(cs_color.value))

            if ts is not None and cs_scores_arr is not None:
                w = cs_weight.value
                scores = (1 - w) * ts + w * cs_scores_arr
                label = f'"{cs_query.value.strip()}" + color {cs_color.value} (w={w:.2f})'
            elif ts is not None:
                scores, label = ts, f'"{cs_query.value.strip()}"'
            else:
                scores, label = cs_scores_arr, f'color {cs_color.value}'

            cs_prog.value = 45; cs_status.value = 'Ranking...'
            idx = np.argsort(-scores)[:cs_n.value].tolist()

            cs_prog.value = 50; cs_status.value = 'Re-sorting by hue...'
            idx.sort(key=lambda i: pixel_feats[i, 1])

            # update state for download
            cs_state['paths']    = [paths[i] for i in idx]
            cs_state['prefixes'] = [f'h{pixel_feats[i,1]:.2f}_' for i in idx]
            cs_state['tag']      = f'colorstory_{label}'.replace(' ', '_')[:60]

            def img_prog(done, total):
                cs_prog.value = 50 + int(50 * done / total)
                cs_status.value = f'Loading thumbnail {done}/{total}'

            swatches = [tuple(dominant_rgb[i]) for i in idx]
            titles = [f'h={pixel_feats[i, 1]:.2f}' for i in idx]
            show_grid_with_swatch(
                [paths[i] for i in idx], swatches=swatches, titles=titles,
                cols=min(10, len(idx)), thumb=140,
                figtitle=f'color story — {label} (sorted by hue)',
                progress_cb=img_prog,
            )
            cs_prog.value = 100; cs_prog.bar_style = 'success'
            cs_status.value = f'Done — {len(idx)} images'
    except Exception as e:
        cs_prog.bar_style = 'danger'; cs_status.value = f'Error: {e}'; raise
    finally:
        cs_btn.disabled = False

cs_btn.on_click(run_color_story)
dl_cs = make_download_button(cs_state)

display(widgets.VBox([
    widgets.HBox([cs_use_text,  cs_query]),
    widgets.HBox([cs_use_color, cs_color]),
    cs_weight,
    widgets.HBox([cs_n, cs_btn]),
    widgets.HBox([cs_prog, cs_status]),
    cs_out,
    dl_cs,
]))

## 9. Bonus — Multi-facet by example image (interactive)

Upload an image. The panel extracts both its **CLIP embedding** and its **dominant colour**, then searches using whichever signals you enable. Same toggle + progress UX as the main panel — pressing **Find similar** is the only thing that triggers a search.

In [9]:
import io
import ipywidgets as widgets
from IPython.display import display

im_state = {'paths': [], 'prefixes': [], 'tag': 'similar'}

im_uploader  = widgets.FileUpload(accept='image/*', multiple=False, description='Upload')
im_use_clip  = widgets.Checkbox(value=True, description='CLIP similarity',
                                indent=False, layout=widgets.Layout(width='180px'))
im_use_color = widgets.Checkbox(value=True, description='Dominant colour',
                                indent=False, layout=widgets.Layout(width='180px'))
im_weight    = widgets.FloatSlider(value=0.3, min=0, max=1, step=0.05,
                                   description='CLIP ↔ Color', readout_format='.2f',
                                   layout=widgets.Layout(width='55%'))
im_topk      = widgets.IntSlider(value=12, min=1, max=40, description='Top K')
im_btn       = widgets.Button(description='Find similar', button_style='primary',
                              icon='search', layout=widgets.Layout(width='200px'))
im_prog      = widgets.IntProgress(value=0, min=0, max=100, bar_style='',
                                   layout=widgets.Layout(width='50%'))
im_status    = widgets.Label(value='Upload an image and press Find similar')
im_out       = widgets.Output()

def run_img_search(_=None):
    im_out.clear_output()
    im_prog.bar_style = 'info'; im_prog.value = 0
    im_status.value = 'Starting...'; im_btn.disabled = True
    try:
        with im_out:
            if not im_uploader.value:
                im_status.value = 'No image uploaded'
                print('Upload an image first.'); return
            if 'embeddings' not in globals():
                print('Build the index first.'); return
            if not (im_use_clip.value or im_use_color.value):
                print('Enable at least one of CLIP or Color.'); return

            im_prog.value = 10; im_status.value = 'Decoding uploaded image...'
            item = (im_uploader.value[0] if isinstance(im_uploader.value, tuple)
                    else next(iter(im_uploader.value.values())))
            q_img = Image.open(io.BytesIO(item['content'])).convert('RGB')
            q_name = item.get('name', 'query')

            ts = cs_scores_arr = None
            q_rgb = None

            if im_use_clip.value:
                im_prog.value = 25; im_status.value = 'CLIP-encoding query image...'
                q_emb = encode_images([q_img])[0]
                s = embeddings @ q_emb
                lo, hi = s.min(), s.max()
                ts = (s - lo) / (hi - lo + 1e-9)

            if im_use_color.value:
                im_prog.value = 35; im_status.value = 'Computing dominant colour...'
                q_rgb = compute_dominant_rgb(q_img)
                cs_scores_arr = color_scores(q_rgb)

            if ts is not None and cs_scores_arr is not None:
                w = im_weight.value
                scores = (1 - w) * ts + w * cs_scores_arr
                label = f'CLIP + color (w={w:.2f})'
            elif ts is not None:
                scores, label = ts, 'CLIP similarity'
            else:
                scores, label = cs_scores_arr, 'color similarity'

            im_prog.value = 45; im_status.value = 'Ranking...'
            idx = np.argsort(-scores)[:im_topk.value].tolist()

            # update state for download
            im_state['paths']    = [paths[i] for i in idx]
            im_state['prefixes'] = [f'{scores[i]:.3f}_' for i in idx]
            im_state['tag']      = f'similar_{Path(q_name).stem}'

            # show query thumbnail
            fig, ax = plt.subplots(figsize=(2.5, 2.5))
            ax.imshow(q_img); ax.axis('off')
            ax.set_title(f'QUERY — {label}\n{_short(q_name)}', fontsize=8)
            if q_rgb is not None:
                w_px, h_px = q_img.size
                ax.add_patch(Rectangle((0, h_px*0.95), w_px, h_px*0.08,
                                       facecolor=q_rgb, edgecolor='black',
                                       linewidth=0.5, clip_on=False))
            plt.show()

            def img_prog(done, total):
                im_prog.value = 50 + int(50 * done / total)
                im_status.value = f'Loading thumbnail {done}/{total}'

            swatches = [tuple(dominant_rgb[i]) for i in idx]
            titles = [f'{scores[i]:.3f}' for i in idx]
            show_grid_with_swatch(
                [paths[i] for i in idx], swatches=swatches, titles=titles,
                cols=5, thumb=180, figtitle=label, progress_cb=img_prog,
            )
            im_prog.value = 100; im_prog.bar_style = 'success'
            im_status.value = f'Done — {len(idx)} matches'
    except Exception as e:
        im_prog.bar_style = 'danger'; im_status.value = f'Error: {e}'; raise
    finally:
        im_btn.disabled = False

im_btn.on_click(run_img_search)
dl_img = make_download_button(im_state)

display(widgets.VBox([
    im_uploader,
    widgets.HBox([im_use_clip, im_use_color]),
    im_weight,
    widgets.HBox([im_topk, im_btn]),
    widgets.HBox([im_prog, im_status]),
    im_out,
    dl_img,
]))